# BERT Fine-Tuning for Medical Claim Verification

This notebook fine-tunes BERT to classify claim-evidence pairs as **Supports**, **Refutes**, or **Neutral**, and compares it against the TF-IDF + Logistic Regression baseline.

BERT is trained twice with identical settings:

| Run | Training data |
|---|---|
| 1 | HealthVer only |
| 2 | HealthVer + HealthFC (combined) |

Every model is evaluated on the same four sets, reported separately: HealthVer dev, HealthVer test, HealthFC dev, HealthFC test.

**How to run (Google Colab):** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`. The full run takes roughly 20-30 minutes on a T4. The notebook clones the repository to get the data, so nothing needs to be uploaded.

## 1. Setup

In [ ]:
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch

REPO_URL = "https://github.com/dong135790/NLP-Final-Project.git"

# Find the data folder: run from notebooks/, from the repo root, or (on Colab) clone the repo.
if Path("../data/healthver_train.csv").exists():
    REPO_DIR = Path("..")
elif Path("data/healthver_train.csv").exists():
    REPO_DIR = Path(".")
else:
    if not Path("NLP-Final-Project").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    REPO_DIR = Path("NLP-Final-Project")

DATA_DIR = REPO_DIR / "data"
RESULTS_DIR = REPO_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Data folder:", DATA_DIR)
print("Device:", device)
if device.type == "cpu":
    print("WARNING: no GPU found. Training on CPU will be very slow.")

In [ ]:
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 256       # tokens for claim + evidence together
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
SEED = 42

# Set to True to check that the whole notebook runs end to end in about a minute.
# It uses a small sample of every dataset and one epoch, so its numbers are meaningless.
QUICK_TEST = False

LABELS = ["Supports", "Refutes", "Neutral"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

## 2. Data

HealthVer comes with official train / dev / test splits.

HealthFC has no official split, so it is split exactly as in `combined_logistic_regression.ipynb`: grouped by article URL with `random_state=42`, giving 596 / 78 / 76 examples. Using the same split keeps the BERT and Logistic Regression results comparable.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

COLUMNS = ["claim", "evidence", "label"]

hv_train = pd.read_csv(DATA_DIR / "healthver_train.csv")[COLUMNS]
hv_dev = pd.read_csv(DATA_DIR / "healthver_dev.csv")[COLUMNS]
hv_test = pd.read_csv(DATA_DIR / "healthver_test.csv")[COLUMNS]

healthfc = pd.read_csv(DATA_DIR / "healthFC.csv", encoding="utf-8-sig")
healthfc = healthfc[["en_claim", "en_top_sentences", "label", "url"]].rename(
    columns={"en_claim": "claim", "en_top_sentences": "evidence"}
)
healthfc["label"] = healthfc["label"].map({0: "Supports", 1: "Neutral", 2: "Refutes"})
healthfc = healthfc.dropna(subset=["claim", "evidence", "label", "url"])

# Same two-step grouped split as combined_logistic_regression.ipynb.
split1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(split1.split(healthfc, groups=healthfc["url"]))
fc_train_full = healthfc.iloc[train_idx]
fc_temp = healthfc.iloc[temp_idx]

split2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
dev_idx, test_idx = next(split2.split(fc_temp, groups=fc_temp["url"]))
fc_dev_full = fc_temp.iloc[dev_idx]
fc_test_full = fc_temp.iloc[test_idx]

assert (len(fc_train_full), len(fc_dev_full), len(fc_test_full)) == (596, 78, 76), \
    "HealthFC split differs from combined_logistic_regression.ipynb"
assert not set(fc_train_full["url"]) & (set(fc_dev_full["url"]) | set(fc_test_full["url"])), \
    "An article appears in both the training and evaluation splits"

fc_train = fc_train_full[COLUMNS].reset_index(drop=True)
fc_dev = fc_dev_full[COLUMNS].reset_index(drop=True)
fc_test = fc_test_full[COLUMNS].reset_index(drop=True)

if QUICK_TEST:
    print("QUICK_TEST is on: using small samples. Do not report these numbers.\n")
    hv_train = hv_train.sample(300, random_state=SEED).reset_index(drop=True)
    hv_dev = hv_dev.sample(100, random_state=SEED).reset_index(drop=True)
    hv_test = hv_test.sample(100, random_state=SEED).reset_index(drop=True)
    fc_train = fc_train.sample(60, random_state=SEED).reset_index(drop=True)
    EPOCHS = 1

combined_train = pd.concat([hv_train, fc_train], ignore_index=True)

TRAIN_SETS = {
    "HealthVer only": hv_train,
    "HealthVer + HealthFC": combined_train,
}
EVAL_SETS = {
    "HealthVer dev": hv_dev,
    "HealthVer test": hv_test,
    "HealthFC dev": fc_dev,
    "HealthFC test": fc_test,
}
# Dev data used to pick the best epoch for each run. Test sets are never used for this.
SELECTION_SETS = {
    "HealthVer only": hv_dev,
    "HealthVer + HealthFC": pd.concat([hv_dev, fc_dev], ignore_index=True),
}

summary = pd.DataFrame(
    {name: df["label"].value_counts().reindex(LABELS) for name, df in {**TRAIN_SETS, **EVAL_SETS}.items()}
).T
summary["Total"] = summary.sum(axis=1)
summary

## 3. Tokenization

The claim and the evidence are passed to BERT as a sentence pair:

```text
[CLS] claim [SEP] evidence [SEP]
```

Pairs longer than `MAX_LENGTH` tokens are truncated. The cell below reports how many examples that affects.

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer)


class PairDataset(torch.utils.data.Dataset):
    # Tokenized claim-evidence pairs with integer labels.

    def __init__(self, df):
        self.encodings = tokenizer(
            df["claim"].tolist(),
            df["evidence"].tolist(),
            truncation=True,
            max_length=MAX_LENGTH,
        )
        self.labels = [LABEL2ID[label] for label in df["label"]]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {key: values[i] for key, values in self.encodings.items()}
        item["labels"] = self.labels[i]
        return item


for name, df in {"HealthVer train": hv_train, "HealthFC train": fc_train}.items():
    lengths = [
        len(ids)
        for ids in tokenizer(df["claim"].tolist(), df["evidence"].tolist(), truncation=False)["input_ids"]
    ]
    too_long = sum(length > MAX_LENGTH for length in lengths)
    print(
        f"{name}: median {int(np.median(lengths))} tokens, max {max(lengths)}, "
        f"truncated {too_long}/{len(lengths)} ({too_long / len(lengths):.1%})"
    )

## 4. Training and evaluation functions

Standard fine-tuning setup: AdamW, linear warmup then linear decay, cross-entropy loss. After each epoch the model is scored on the dev data and the epoch with the best macro-F1 is kept.

Unlike the Logistic Regression baseline (`class_weight="balanced"`), the loss here is not class-weighted. Macro-F1 is the main metric because it weights the three classes equally.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup


def predict_bert(model, df):
    # Returns the predicted label (string) for every row of df.
    loader = torch.utils.data.DataLoader(PairDataset(df), batch_size=64, collate_fn=collator)
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            predictions.extend(model(**batch).logits.argmax(dim=-1).tolist())
    return [LABELS[i] for i in predictions]


def train_bert(train_df, selection_df, run_name):
    # Fine-tunes a fresh BERT on train_df and returns the weights from the best dev epoch.
    set_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(LABELS)).to(device)

    loader = torch.utils.data.DataLoader(
        PairDataset(train_df),
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        generator=torch.Generator().manual_seed(SEED),
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = len(loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)

    best_f1, best_state, history = -1.0, None, []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        for batch in tqdm(loader, desc=f"{run_name} | epoch {epoch}/{EPOCHS}", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            loss = model(**batch).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            running_loss += loss.item()

        dev_f1 = f1_score(selection_df["label"], predict_bert(model, selection_df), average="macro")
        train_loss = running_loss / len(loader)
        history.append({"run": run_name, "epoch": epoch, "train_loss": train_loss, "dev_macro_f1": dev_f1})
        print(f"{run_name} | epoch {epoch}: train loss {train_loss:.4f}, dev macro-F1 {dev_f1:.4f}")

        if dev_f1 > best_f1:
            best_f1 = dev_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model, history


results = []       # one row per (model, training data, evaluation set)
predictions = {}   # (model, training data, evaluation set) -> list of predicted labels


def record(model_name, train_name, eval_name, y_true, y_pred):
    report = classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0)
    results.append({
        "model": model_name,
        "train_data": train_name,
        "eval_set": eval_name,
        "n": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": report["macro avg"]["f1-score"],
        **{f"f1_{label}": report[label]["f1-score"] for label in LABELS},
        **{f"precision_{label}": report[label]["precision"] for label in LABELS},
        **{f"recall_{label}": report[label]["recall"] for label in LABELS},
    })
    predictions[(model_name, train_name, eval_name)] = list(y_pred)

## 5. Fine-tune BERT

Two runs with identical settings. Only the training data changes.

In [ ]:
training_history = []

for train_name, train_df in TRAIN_SETS.items():
    print(f"=== BERT trained on {train_name} ({len(train_df)} examples) ===")
    model, history = train_bert(train_df, SELECTION_SETS[train_name], train_name)
    training_history.extend(history)

    for eval_name, eval_df in EVAL_SETS.items():
        y_pred = predict_bert(model, eval_df)
        record("BERT", train_name, eval_name, eval_df["label"], y_pred)
        if eval_name.endswith("test"):
            print(f"\n{eval_name}")
            print(classification_report(eval_df["label"], y_pred, labels=LABELS, zero_division=0))

    # Free GPU memory before the next run.
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

pd.DataFrame(training_history)

## 6. Logistic Regression baseline on the same data

The existing notebooks evaluate the combined Logistic Regression model on HealthFC dev only. To compare it with BERT, the cell below refits the same pipeline (TF-IDF with 1-2 grams, 10,000 features, balanced class weights) on the same two training sets and scores it on the same four evaluation sets.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


def combine_text(df):
    return "Claim: " + df["claim"] + " Evidence: " + df["evidence"]


for train_name, train_df in TRAIN_SETS.items():
    lr_model = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])
    lr_model.fit(combine_text(train_df), train_df["label"])

    for eval_name, eval_df in EVAL_SETS.items():
        record("TF-IDF + LR", train_name, eval_name, eval_df["label"], lr_model.predict(combine_text(eval_df)))

## 7. Results

HealthFC dev and test have fewer than 80 examples each, so one example is worth about 1.3 accuracy points there. Treat small differences on HealthFC as noise and base the main conclusions on HealthVer test.

In [ ]:
results_df = pd.DataFrame(results)
eval_order = list(EVAL_SETS)


def comparison_table(metric):
    table = results_df.pivot_table(index=["model", "train_data"], columns="eval_set", values=metric)
    return table[eval_order].round(3)


print("Macro-F1")
display(comparison_table("macro_f1"))
print("Accuracy")
display(comparison_table("accuracy"))

In [ ]:
print("Per-class F1 on the test sets")
per_class = results_df[results_df["eval_set"].str.endswith("test")]
per_class.set_index(["eval_set", "model", "train_data"])[[f"f1_{label}" for label in LABELS]].round(3).sort_index()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Confusion matrices on HealthVer test for the models trained on the combined data.
TRAIN_FOR_PLOTS = "HealthVer + HealthFC"
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, model_name in zip(axes, ["TF-IDF + LR", "BERT"]):
    ConfusionMatrixDisplay.from_predictions(
        hv_test["label"],
        predictions[(model_name, TRAIN_FOR_PLOTS, "HealthVer test")],
        labels=LABELS,
        cmap="Blues",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(f"{model_name} (HealthVer test)")
plt.tight_layout()
plt.show()

## 8. Error analysis

The Logistic Regression notebook found that the baseline often confuses **Refutes** with **Supports**, which is the most harmful mistake for fact-checking. The cells below check whether BERT fixes those cases and show examples BERT still gets wrong.

In [ ]:
analysis = hv_test.copy()
analysis["lr_pred"] = predictions[("TF-IDF + LR", TRAIN_FOR_PLOTS, "HealthVer test")]
analysis["bert_pred"] = predictions[("BERT", TRAIN_FOR_PLOTS, "HealthVer test")]

lr_correct = analysis["lr_pred"] == analysis["label"]
bert_correct = analysis["bert_pred"] == analysis["label"]

print("Agreement between the two models on HealthVer test")
display(pd.crosstab(
    lr_correct.map({True: "LR correct", False: "LR wrong"}),
    bert_correct.map({True: "BERT correct", False: "BERT wrong"}),
))

for wrong, right in [("Supports", "Refutes"), ("Refutes", "Supports")]:
    is_true = analysis["label"] == right
    lr_n = (is_true & (analysis["lr_pred"] == wrong)).sum()
    bert_n = (is_true & (analysis["bert_pred"] == wrong)).sum()
    print(f"True {right} predicted as {wrong}: LR {lr_n}, BERT {bert_n} (of {is_true.sum()})")

In [ ]:
pd.set_option("display.max_colwidth", None)

print("True Refutes that BERT predicted as Supports (first 5)")
display(analysis[(analysis["label"] == "Refutes") & (analysis["bert_pred"] == "Supports")].head(5))

print("Examples the baseline got wrong and BERT got right (first 5)")
display(analysis[~lr_correct & bert_correct].head(5))

## 9. Save results

Colab deletes its files when the session ends, so the metrics, the test-set predictions and the training history are written to `results/` and offered as a download.

In [ ]:
suffix = "_quicktest" if QUICK_TEST else ""

results_path = RESULTS_DIR / f"bert_vs_lr_results{suffix}.csv"
results_df.to_csv(results_path, index=False)

rows = []
for (model_name, train_name, eval_name), y_pred in predictions.items():
    if not eval_name.endswith("test"):
        continue
    out = EVAL_SETS[eval_name].copy()
    out.insert(0, "eval_set", eval_name)
    out.insert(0, "train_data", train_name)
    out.insert(0, "model", model_name)
    out["prediction"] = y_pred
    rows.append(out)
predictions_path = RESULTS_DIR / f"bert_vs_lr_predictions{suffix}.csv"
pd.concat(rows, ignore_index=True).to_csv(predictions_path, index=False)

history_path = RESULTS_DIR / f"bert_training_history{suffix}.csv"
pd.DataFrame(training_history).to_csv(history_path, index=False)

print("Saved:", results_path, predictions_path, history_path)

try:
    from google.colab import files
    files.download(str(results_path))
    files.download(str(predictions_path))
    files.download(str(history_path))
except ImportError:
    pass